In [ ]:
!pip install torchsummary
!pip install torchinfo

# Model with picobackbone

In [5]:
import torch
from torchinfo import summary
from nets import pico


model = pico.yolo_v8_p(80).cuda()

# Create a dummy input tensor
dummy_input = torch.randn(1, 3, 640, 640).cuda()

# Display the model summary
summary(model, input_size=(1, 3, 640, 640), device='cuda')

Layer (type:depth-idx)                                  Output Shape              Param #
YOLO                                                    [1, 84, 8400]             --
├─PicoBackbone: 1-1                                     [1, 32, 80, 80]           --
│    └─ESNet: 2-1                                       [1, 32, 80, 80]           --
│    │    └─ConvBNLayer: 3-1                            [1, 24, 320, 320]         696
│    │    └─ModuleList: 3-4                             --                        (recursive)
│    │    └─MaxPool2d: 3-3                              [1, 24, 160, 160]         --
│    │    └─ModuleList: 3-4                             --                        (recursive)
├─DarkFPN: 1-2                                          [1, 32, 80, 80]           --
│    └─Upsample: 2-2                                    [1, 128, 40, 40]          --
│    └─CSP: 2-3                                         [1, 64, 40, 40]           --
│    │    └─Conv: 3-5                    

## Model with darknet


In [3]:
import torch
from torchsummary import summary
from nets import darknet

class BackboneWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Create YOLO v8 nano model and get its backbone
        self.model = darknet.yolo_v8_n()
        self.backbone = self.model.net
    
    def forward(self, x):
        # Return all feature maps from backbone
        return self.backbone(x)


# Create and move model to GPU
model = BackboneWrapper().cuda()

summary(model, (3, 640, 640))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 16, 320, 320]             432
            Conv2d-2         [-1, 16, 320, 320]             432
       BatchNorm2d-3         [-1, 16, 320, 320]              32
       BatchNorm2d-4         [-1, 16, 320, 320]              32
              SiLU-5         [-1, 16, 320, 320]               0
              SiLU-6         [-1, 16, 320, 320]               0
              Conv-7         [-1, 16, 320, 320]               0
              Conv-8         [-1, 16, 320, 320]               0
            Conv2d-9         [-1, 32, 160, 160]           4,608
           Conv2d-10         [-1, 32, 160, 160]           4,608
      BatchNorm2d-11         [-1, 32, 160, 160]              64
      BatchNorm2d-12         [-1, 32, 160, 160]              64
             SiLU-13         [-1, 32, 160, 160]               0
             SiLU-14         [-1, 32, 1

## Tiny Yolo

In [2]:
import torch
from torchinfo import summary
from nets import tiny

class ModelWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = tiny.YOLOTinyFPGA(num_classes=80)
    
    def forward(self, x):
        # Force training mode output
        self.model.train()
        return self.model(x)

# Create wrapped model and move to GPU
model = ModelWrapper().cuda()

# Display model summary with torchinfo
summary(model, 
        input_size=(1, 3, 640, 640),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        depth=4,
        device="cuda")

Layer (type:depth-idx)                             Input Shape               Output Shape              Param #                   Trainable
ModelWrapper                                       [1, 3, 640, 640]          [1, 144, 80, 80]          --                        True
├─YOLOTinyFPGA: 1-1                                [1, 3, 640, 640]          [1, 144, 80, 80]          --                        True
│    └─MinimalBackbone: 2-1                        [1, 3, 640, 640]          [1, 8, 80, 80]            --                        True
│    │    └─Conv1x1: 3-1                           [1, 3, 640, 640]          [1, 8, 320, 320]          --                        True
│    │    │    └─Conv2d: 4-1                       [1, 3, 640, 640]          [1, 8, 320, 320]          24                        True
│    │    │    └─BatchNorm2d: 4-2                  [1, 8, 320, 320]          [1, 8, 320, 320]          16                        True
│    │    │    └─SiLU: 4-3                         [1, 8,

## Tinysimo Yolo

In [2]:
import torch
from torchsummary import summary
from nets import tinysimo

class BackboneWrapper(torch.nn.Module):
    def __init__(self):
        super().__init__()
        # Create YOLO v8 nano model and get its backbone
        self.model = tinysimo.yolo_v8()
        self.backbone = self.model.net
    
    def forward(self, x):
        # Return all feature maps from backbone
        return self.backbone(x)


# Create and move model to GPU
model = BackboneWrapper().cuda()

summary(model, (3, 640, 640))


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 16, 640, 640]             432
            Conv2d-2         [-1, 16, 640, 640]             432
       BatchNorm2d-3         [-1, 16, 640, 640]              32
       BatchNorm2d-4         [-1, 16, 640, 640]              32
              SiLU-5         [-1, 16, 640, 640]               0
              SiLU-6         [-1, 16, 640, 640]               0
              Conv-7         [-1, 16, 640, 640]               0
              Conv-8         [-1, 16, 640, 640]               0
         MaxPool2d-9         [-1, 16, 320, 320]               0
        MaxPool2d-10         [-1, 16, 320, 320]               0
           Conv2d-11         [-1, 16, 320, 320]           2,304
           Conv2d-12         [-1, 16, 320, 320]           2,304
      BatchNorm2d-13         [-1, 16, 320, 320]              32
      BatchNorm2d-14         [-1, 16, 3